# Train the first pass: which sentences are eligibility requirements?

Frozen MiniLM embeddings + one logistic regression per label
(location, business_type, audience). MiniLM itself is not changed; the
classifiers learn what each kind of requirement looks like in its vector
space.

Folder layout (next to this notebook):
    grant_utils.py
    grants/   grant .txt files; the file name (without .txt) is the grant_id
    labels/   sentences.csv (your labels), to_label.csv (made by Cell 3)
    models/   first_pass.joblib (made by Cell 7)

Workflow:
  1. Run Cells 1-3 to make labels/to_label.csv, pre-filled by zero-shot.
  2. Correct it by hand (check every row, not just the 1s), then append
     the rows to labels/sentences.csv. In Excel: Save As "CSV UTF-8".
  3. Run Cells 4-7 to train, compare against zero-shot, and save.

In [1]:
# One-time setup: run this, then Kernel -> Restart. Skip it after that.
%pip install --user sentence-transformers scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---------------------------------------------------------------------------
# CELL 1: imports and settings
# ---------------------------------------------------------------------------
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import GroupKFold

from grant_utils import (
    EMBED_MODEL, LABEL_COLUMNS, QUERIES,
    embed, read_grant, split_sentences, zero_shot_scores,
)

GRANTS_DIR = Path("grants")
LABEL_CSV = Path("labels/sentences.csv")
TO_LABEL_CSV = Path("labels/to_label.csv")
MODEL_OUT = Path("models/first_pass.joblib")

# Held-out test grants (grant_ids). These are never trained on.
# Leave empty and Cell 4 picks ~20% at random and prints them: paste that
# list here so the test set stays the same every run.
TEST_GRANTS = []

TARGET_RECALL = 0.90      # threshold is set so ~90% of requirements get caught
ZERO_SHOT_MIN_SIM = 0.30  # same cutoff the zero-shot first pass uses
MIN_POSITIVES = 5         # labels with fewer training examples stay zero-shot

In [3]:
# ---------------------------------------------------------------------------
# CELL 2: load the embedding model (run once)
# ---------------------------------------------------------------------------
embedder = SentenceTransformer(EMBED_MODEL)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# ---------------------------------------------------------------------------
# CELL 3: make a labeling sheet for grants that aren't labeled yet.
# Pre-filled with zero-shot guesses (1 = looks like that kind of requirement).
# ---------------------------------------------------------------------------
already_labeled = set()
if LABEL_CSV.exists():
    already_labeled = set(
        pd.read_csv(LABEL_CSV, encoding="utf-8-sig")["grant_id"].astype(str)
    )

rows = []
for path in sorted(GRANTS_DIR.glob("*.txt")):
    grant_id = path.stem
    if grant_id in already_labeled:
        continue
    sentences = split_sentences(read_grant(path))
    if not sentences:
        continue
    vecs = embed(embedder, sentences)
    guesses = {
        col: zero_shot_scores(embedder, vecs, QUERIES[col]) >= ZERO_SHOT_MIN_SIM
        for col in LABEL_COLUMNS
    }
    for i, sentence in enumerate(sentences):
        rows.append({
            "grant_id": grant_id,
            "sentence": sentence,
            **{col: int(guesses[col][i]) for col in LABEL_COLUMNS},
        })

if rows:
    TO_LABEL_CSV.parent.mkdir(parents=True, exist_ok=True)
    sheet = pd.DataFrame(rows)
    sheet.to_csv(TO_LABEL_CSV, index=False, encoding="utf-8-sig")
    print(f"Wrote {len(sheet)} sentences from {sheet.grant_id.nunique()} "
          f"grants to {TO_LABEL_CSV}")
else:
    print("Every grant in grants/ is already labeled.")

Every grant in grants/ is already labeled.


In [10]:
# ---------------------------------------------------------------------------
# CELL 4, loads the labels n stuff
# ---------------------------------------------------------------------------
if not LABEL_CSV.exists():
    raise FileNotFoundError(
        f"No labels yet. Correct {TO_LABEL_CSV}, then save it as {LABEL_CSV}."
    )
df = pd.read_csv(LABEL_CSV, encoding="utf-8-sig")
missing = {"grant_id", "sentence", *LABEL_COLUMNS} - set(df.columns)
if missing:
    raise ValueError(f"{LABEL_CSV} is missing columns: {missing}")

df = df.dropna(subset=["sentence"]).reset_index(drop=True)
df["grant_id"] = df["grant_id"].astype(str)
df[LABEL_COLUMNS] = df[LABEL_COLUMNS].fillna(0).astype(int)

all_grants = sorted(df.grant_id.unique())
if not TEST_GRANTS:
    rng = random.Random(42)
    TEST_GRANTS = sorted(rng.sample(all_grants, max(1, round(0.2 * len(all_grants)))))
    print("No TEST_GRANTS set. Picked:", TEST_GRANTS)
    print("Paste that list into Cell 1 so it stays fixed.\n")
unknown = set(TEST_GRANTS) - set(all_grants)
if unknown:
    print("Warning: these TEST_GRANTS have no labeled rows:", sorted(unknown))

vecs = embed(embedder, df.sentence)
is_test = df.grant_id.isin(TEST_GRANTS).to_numpy()
train_df, test_df = df[~is_test], df[is_test]
X_train, X_test = vecs[~is_test], vecs[is_test]
train_groups = train_df.grant_id.to_numpy()

print(f"Train: {train_df.grant_id.nunique()} grants, {len(train_df)} sentences")
print(f"Test:  {test_df.grant_id.nunique()} grants, {len(test_df)} sentences")
print("Positive sentences (train / test):")
for col in LABEL_COLUMNS:
    print(f"  {col:<14} {train_df[col].sum():>4} / {test_df[col].sum()}")

Train: 6 grants, 104 sentences
Test:  2 grants, 90 sentences
Positive sentences (train / test):
  location         30 / 14
  business_type    48 / 37
  audience         27 / 27


In [11]:
# ---------------------------------------------------------------------------
# CELL 5: train one classifier per label.
# The threshold is chosen from out-of-fold predictions (each training grant
# is scored by a model that never saw it), never from the test set.
# ---------------------------------------------------------------------------
def make_classifier(): # currently we have 3 classifiers: location, business audience. 
    # class_weight="balanced" because most sentences are not requirements
    return LogisticRegression(class_weight="balanced", max_iter=2000) #how many times we can loop through this training 


def out_of_fold_probs(X, y, groups): # this sets aside one of the grants to be tested on while the other grant ids are used for training
    """Score every training sentence with a model trained on the OTHER grants."""
    n_splits = min(5, len(set(groups)))
    if n_splits < 2:
        raise ValueError("Need at least 2 training grants.") # error that pops up if theres not enough training grant docs 
    probs = np.full(len(y), np.nan)
    for fit_idx, score_idx in GroupKFold(n_splits=n_splits).split(X, y, groups):
        if len(set(y[fit_idx])) < 2:  # this fold has no positives to learn from
            continue
        clf = make_classifier().fit(X[fit_idx], y[fit_idx])
        probs[score_idx] = clf.predict_proba(X[score_idx])[:, 1]
    return probs


def pick_threshold(y, probs, target_recall): # sort of like when you curve grades, this sets a threshold based on all the scores w/ relation to recall so you don't lose every sentence 
    """Highest threshold that still catches target_recall of the positives."""
    pos = np.sort(probs[(y == 1) & ~np.isnan(probs)])
    k = int(np.floor((1 - target_recall) * len(pos)))
    return float(pos[k])


first_pass = {}
for col in LABEL_COLUMNS:
    y = train_df[col].to_numpy()
    n_pos = int(y.sum())
    if n_pos < MIN_POSITIVES:
        print(f"{col}: only {n_pos} positives, skipped (stays zero-shot)")
        continue
    oof = out_of_fold_probs(X_train, y, train_groups)
    threshold = pick_threshold(y, oof, TARGET_RECALL)
    clf = make_classifier().fit(X_train, y)
    first_pass[col] = {"clf": clf, "threshold": threshold}
    print(f"{col}: trained on {n_pos} positives, threshold {threshold:.3f}")

location: trained on 30 positives, threshold 0.391
business_type: trained on 48 positives, threshold 0.434
audience: trained on 27 positives, threshold 0.374


In [13]:
# ---------------------------------------------------------------------------
# CELL 6: test-set results, trained vs. zero-shot.
# Recall matters most: a requirement missed here never reaches the NLI step.
# ---------------------------------------------------------------------------
from sklearn.metrics import accuracy_score, f1_score


def scores(y, pred):
    return {
        "recall":    recall_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "f1":        f1_score(y, pred, zero_division=0),
        "accuracy":  accuracy_score(y, pred),
    }


def show(name, s):
    print(f"  {name:<10} recall {s['recall']:.2f}  precision {s['precision']:.2f}  "
          f"f1 {s['f1']:.2f}  accuracy {s['accuracy']:.2f}")


for col in LABEL_COLUMNS:
    y = test_df[col].to_numpy()
    print("\n" + "=" * 70)
    print(f"{col}   ({int(y.sum())} of {len(y)} test sentences are real requirements)")
    if y.sum() == 0:
        print("  No positive sentences in the test grants, nothing to measure.")
        continue

    zs_pred = zero_shot_scores(embedder, X_test, QUERIES[col]) >= ZERO_SHOT_MIN_SIM
    show("zero-shot", scores(y, zs_pred))

    if col not in first_pass:
        print("  (no trained model for this label)")
        continue
    entry = first_pass[col]
    probs = entry["clf"].predict_proba(X_test)[:, 1]
    pred = probs >= entry["threshold"]
    show("trained", scores(y, pred))

    missed = test_df.sentence[(y == 1) & ~pred]
    if len(missed):
        print(f"  Missed by trained model ({len(missed)}):")
        for s in missed.head(5):
            print(f'    - "{s}"')


location   (14 of 90 test sentences are real requirements)
  zero-shot  recall 1.00  precision 0.58  f1 0.74  accuracy 0.89
  trained    recall 0.86  precision 0.57  f1 0.69  accuracy 0.88
  Missed by trained model (2):
    - "impact in the targeted application areas."
    - "Choosing stages (a), (b) or (c) will make your application ineligible as this would"

business_type   (37 of 90 test sentences are real requirements)
  zero-shot  recall 0.46  precision 0.81  f1 0.59  accuracy 0.73
  trained    recall 0.46  precision 0.89  f1 0.61  accuracy 0.76
  Missed by trained model (20):
    - "Women TechEU targets early-stage women-led deep tech startups."
    - "also currently hold a top management position (CEO, CTO, CSO or equivalent)"
    - "of the shares in the CAP table (capitalisation table) of the company."
    - "maximum of five years at the time of the eligibility cut-off date they are"
    - "balance sheet total not exceeding EUR 43 million"

audience   (27 of 90 test sentences 

In [14]:
# ---------------------------------------------------------------------------
# CELL 7: save. Only save if the trained results beat zero-shot in Cell 6.
# grant_eligibility uses a trained classifier for each label saved here
# and zero-shot for the rest.

#aka, saves the weights once they've been deemed worthy (and actually useful) 
# ---------------------------------------------------------------------------
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "embed_model": EMBED_MODEL,
        "labels": first_pass,
        "train_grants": sorted(set(train_groups)),
        "test_grants": TEST_GRANTS,
        "target_recall": TARGET_RECALL,
    },
    MODEL_OUT,
)
print(f"Saved {sorted(first_pass)} to {MODEL_OUT}")

Saved ['audience', 'business_type', 'location'] to models/first_pass.joblib


In [15]:
from pathlib import Path
print("Notebook is running in:", Path.cwd())
labels = Path("labels")
print("labels/ contains:", sorted(p.name for p in labels.iterdir()) if labels.exists() else "NO labels/ folder here")
print("Any csv/xlsx nearby:", [str(p) for p in Path(".").glob("**/*") if p.suffix.lower() in (".csv", ".xlsx")])

Notebook is running in: /WAVE/users2/unix/nwiesen
labels/ contains: ['.ipynb_checkpoints', 'sentences.csv', 'to_label.csv']
Any csv/xlsx nearby: ['labels/to_label.csv', 'labels/sentences.csv', 'labels/.ipynb_checkpoints/to_label-checkpoint.csv', '.local/share/Trash/files/sentences.csv']
